# Tech Challenge — Classificação da Qualidade de Vinhos

Notebook principal do projeto. Execute todas as células para reproduzir a análise, os gráficos e os modelos.

## 1. Problema e variável-alvo

Classificar vinhos como alta qualidade (`quality >= 7`) ou baixa/média qualidade (`quality < 7`) usando variáveis físico-químicas.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import *
from sklearn.inspection import permutation_importance
RANDOM_STATE=42
PROJECT_ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
RESULTS_DIR=PROJECT_ROOT/'results'; RESULTS_DIR.mkdir(exist_ok=True)

## 2. Carregamento dos dados

In [ ]:
paths=[Path('../data/WineQT.csv'),Path('data/WineQT.csv'),Path('WineQT.csv'),Path('/mnt/data/WineQT.csv')]
data_path=next((p for p in paths if p.exists()),None)
if data_path is None: raise FileNotFoundError('WineQT.csv não encontrado. Faça upload do arquivo no Colab.')
df=pd.read_csv(data_path)
print('Dimensões:',df.shape)
display(df.head())
display(pd.DataFrame({'tipo':df.dtypes.astype(str),'nulos':df.isna().sum(),'valores_unicos':df.nunique()}))

## 3. Variável-alvo e balanceamento

In [ ]:
df['high_quality']=(df['quality']>=7).astype(int)
class_dist=df['high_quality'].value_counts().sort_index().rename(index={0:'Baixa/Média',1:'Alta'}).to_frame('quantidade')
class_dist['percentual']=(class_dist['quantidade']/len(df)*100).round(2)
display(class_dist)
ax=class_dist['quantidade'].plot(kind='bar',figsize=(7,4),title='Balanceamento da variável-alvo'); ax.tick_params(axis='x',rotation=0); plt.tight_layout(); plt.show()

## 4. EDA — estatísticas, distribuições, outliers e correlações

In [ ]:
features0=[c for c in df.columns if c not in ['Id','quality','high_quality']]
display(df[features0+['quality']].describe().T)
df['quality'].value_counts().sort_index().plot(kind='bar',figsize=(8,4),title='Distribuição das notas'); plt.tight_layout(); plt.show()
df[features0].hist(figsize=(16,12),bins=25); plt.tight_layout(); plt.show()

In [ ]:
out=[]
for col in features0:
    q1,q3=df[col].quantile([.25,.75]); iqr=q3-q1; lo,hi=q1-1.5*iqr,q3+1.5*iqr
    count=((df[col]<lo)|(df[col]>hi)).sum(); out.append([col,lo,hi,count,round(count/len(df)*100,2)])
display(pd.DataFrame(out,columns=['variavel','limite_inferior','limite_superior','possiveis_outliers','percentual']).sort_values('percentual',ascending=False))

In [ ]:
class_means=df.groupby('high_quality')[features0].mean().T.rename(columns={0:'Baixa/Média',1:'Alta'})
class_means['dif_%_alta_vs_baixa']=((class_means['Alta']/class_means['Baixa/Média']-1)*100).round(2)
display(class_means.sort_values('dif_%_alta_vs_baixa',ascending=False))
corr=df[features0+['quality','high_quality']].corr(); display(corr['high_quality'].drop(['quality','high_quality']).sort_values(key=abs,ascending=False).to_frame('correlacao'))

## 5. Feature engineering e preparação dos dados

In [ ]:
model_df=df.copy()
model_df['bound_sulfur_dioxide']=model_df['total sulfur dioxide']-model_df['free sulfur dioxide']
model_df['free_sulfur_ratio']=model_df['free sulfur dioxide']/(model_df['total sulfur dioxide']+1e-6)
model_df['acidity_ratio']=model_df['fixed acidity']/(model_df['volatile acidity']+1e-6)
model_df['alcohol_density_ratio']=model_df['alcohol']/(model_df['density']+1e-6)
features=[c for c in model_df.columns if c not in ['Id','quality','high_quality']]
X=model_df[features]; y=model_df['high_quality']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=RANDOM_STATE)
print(X_train.shape,X_test.shape)

## 6. Modelos, validação cruzada e avaliação

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
models={'Baseline':(DummyClassifier(strategy='most_frequent'),{}),'Regressão Logística':(Pipeline([('scaler',StandardScaler()),('model',LogisticRegression(class_weight='balanced',max_iter=3000,random_state=RANDOM_STATE))]),{'model__C':[.1,1,10]}),'Random Forest':(RandomForestClassifier(class_weight='balanced',random_state=RANDOM_STATE,n_jobs=1),{'n_estimators':[200],'max_depth':[None,8],'min_samples_leaf':[1,3]})}
fitted={}; rows=[]
for name,(est,params) in models.items():
    if params:
        gs=GridSearchCV(est,params,scoring='roc_auc',cv=cv,n_jobs=1,refit=True); gs.fit(X_train,y_train); fitted[name]=gs.best_estimator_; print(name,gs.best_score_,gs.best_params_)
    else:
        est.fit(X_train,y_train); fitted[name]=est
for name,m in fitted.items():
    yp=m.predict(X_test); prob=m.predict_proba(X_test)[:,1] if hasattr(m,'predict_proba') else yp
    rows.append({'modelo':name,'accuracy':accuracy_score(y_test,yp),'precision':precision_score(y_test,yp,zero_division=0),'recall':recall_score(y_test,yp,zero_division=0),'f1':f1_score(y_test,yp,zero_division=0),'roc_auc':roc_auc_score(y_test,prob)})
metrics=pd.DataFrame(rows).set_index('modelo').sort_values('roc_auc',ascending=False); display(metrics.round(3))

## 7. Matrizes, curvas e interpretação

In [ ]:
for name,m in fitted.items():
    if name!='Baseline':
        ConfusionMatrixDisplay.from_predictions(y_test,m.predict(X_test),display_labels=['Baixa/Média','Alta'],values_format='d'); plt.title(name); plt.show()
fig,ax=plt.subplots(figsize=(8,6))
for name,m in fitted.items():
    if name!='Baseline': RocCurveDisplay.from_estimator(m,X_test,y_test,name=name,ax=ax)
ax.plot([0,1],[0,1],'--'); plt.show()

In [ ]:
best=metrics.index[0] if metrics.index[0]!='Baseline' else metrics.drop(index='Baseline').index[0]
perm=permutation_importance(fitted[best],X_test,y_test,scoring='roc_auc',n_repeats=10,random_state=RANDOM_STATE,n_jobs=1)
imp=pd.DataFrame({'variavel':features,'importancia_media':perm.importances_mean,'desvio':perm.importances_std}).sort_values('importancia_media',ascending=False)
display(imp.head(15))
imp.head(12).sort_values('importancia_media').plot(x='variavel',y='importancia_media',xerr='desvio',kind='barh',figsize=(9,6),legend=False,title=f'Importância — {best}'); plt.tight_layout(); plt.show()

## 8. Conclusão

Os modelos supervisionados superam o baseline e demonstram que características físico-químicas podem apoiar a triagem de vinhos de maior qualidade. O modelo deve ser usado como apoio à decisão, validado com novas amostras e interpretado em conjunto com especialistas.